# Импорты

In [8]:
from typing import List

import numpy as np
import scipy.stats as sps
from scipy.special import gamma, hyp2f1

# 1. Реализация функции z-теста и t-теста

Если дисперсия известна, то применяется z-тест: <br>
$\displaystyle
H_0: \mu = \mu_0 \\
H_1: \mu \neq \mu_0;~ \mu > \mu_0;~ \mu < \mu_0 \\
z = \frac{(\overline{X} - \mu_0) \sqrt{n}}{\sigma} \\
p = \begin{cases}
    1 - \Phi(z), & H_1: \mu > \mu_0 \\
    \Phi(z), & H_1: \mu < \mu_0 \\
    2\Phi(-|z|), & H_1: \mu \neq \mu_0
\end{cases} \\
\Phi(z) = \frac{1}{\sqrt{2\pi}} \int_0^x e^{-\frac{t^2}{2}} dt
$

Если дисперсия неизвестна, то применяется t-тест: <br>
$\displaystyle
H_0: \mu = \mu_0 \\
H_1: \mu \neq \mu_0;~ \mu > \mu_0;~ \mu < \mu_0 \\
t = \frac{(\overline{X} - \mu_0) \sqrt{n}}{\hat{\sigma}_X} \\
\hat{\sigma}_X = \sqrt{\sum_{i = 1}^n \frac{(X_i - \overline{X})^2}{n - 1}} \\
p = \begin{cases}
    1 - F_{T_{n - 1}}(t), & H_1: \mu > \mu_0 \\
    F_{T_{n - 1}}(t), & H_1: \mu < \mu_0 \\
    2F_{T_{n - 1}}(-|t|), & H_1: \mu \neq \mu_0
\end{cases} \\
F_{T_{n}} = \frac{1}{2} + x\Gamma \left( \frac{n + 1}{2} \right) \cdot \frac{\displaystyle {}_2F_1 \left( \frac{1}{2};~ \frac{n + 1}{2};~ \frac{3}{2} ;~ -\frac{x^2}{n} \right)}{\displaystyle \sqrt{\pi n} \Gamma \left( \frac{n}{2} \right)}
$

In [9]:
def test_mean(X_n: List[float], mu_0: float, alternative: str, std: float = None) -> float:
    def Phi(x: float) -> float:
        return sps.norm.cdf(x) - 0.5

    def F_T_n(x: float, n: int) -> float:
        return 0.5 + x * gamma((n + 1) / 2) * hyp2f1(0.5, (n + 1) / 2, 1.5, -x ** 2 / n) / (np.sqrt(np.pi * n) * gamma(n / 2))


    n = len(X_n)
    X_mean = sum(X_n) / n
    if std is None:
        std = np.sqrt(sum((X - X_mean) ** 2 for X in X_n) / (n - 1))
        F = lambda x: F_T_n(x, n - 1)
    else:
        F = Phi

    t = (X_mean - mu_0) * np.sqrt(n) / std

    match alternative:
        case "less":
            return F(t)
        case "greater":
            return 1 - F(t)
        case "two-sided":
            return 2 * F(-np.abs(t))
        case _:
            raise ValueError(f"Unknown alternative: {alternative}")

# 2. Задача 1

Установлено, что средний вес таблетки лекарства сильного действия должен быть равен $a_0 = 0,50$ мг. Выборочная проверка 121 таблетки полученной партии лекарства показала, что средний вес таблетки этой партии $\overline{x} = 0,53$ мг. Требуется при уровне значимости $0,01$ проверить нулевую гипотезу $H_0: a = a_0 = 0,50$ при конкурирующей гипотезе $H_1: a > 0,50$. Многократными предварительными опытами по взвешиванию таблеток, представляемых фармацевтическим заводом, было установлено, что вес таблеток распределён нормально со средним квадратичным отклонением $\sigma = 0,11$ мг.

In [10]:
pvalue = test_mean([0.53] * 121, 0.5, "greater", 0.11)
print(f"p-значение = {pvalue}, выполнение H_0: {pvalue > 0.01}")

p-значение = 0.5013498980316301, выполнение H_0: True


Так как $p$-значение $> \alpha = 0.01$, то гипотезу $H_0$ принимаем

# 3. Задача 2

Проектный контролируемый размер изделий, изготовляемых станком-автоматом, $a = a_0 = 35$ мм. Измерения 20 случайно отобранных изделий дали следующие результаты:
| контролируемый размер $x_i$ | частота (число изделий) $n_i$ |
| --- | --- |
| 34,8 | 2 |
| 34,9 | 3 |
| 35,0 | 4 |
| 35,1 | 6 |
| 35,3 | 5 |

Требуется при уровне значимости $0,05$ проверить нулевую гипотезу $H_0: a = a_0 = 35$ при конкурирующей гипотезе $H_1: a \neq 35$

In [11]:
X_n = [34.8] * 2 + [34.9] * 3 + [35] * 4 + [35.1] * 6 + [35.3] * 5

pvalue = test_mean(X_n, 35, "two-sided")
print(f"Моя: p-значение = {pvalue}, выполнение H_0: {pvalue > 0.05}")

_, pvalue = sps.ttest_1samp(X_n, popmean=35)
print(f"SciPy: p-значение = {pvalue}, выполнение H_0: {pvalue > 0.05}")

Моя: p-значение = 0.07430101762236851, выполнение H_0: True
SciPy: p-значение = 0.07430101762239591, выполнение H_0: True


Так как $p$-значение $> \alpha = 0.05$, то гипотезу $H_0$ принимаем